# 03 - Violent vs Non-Violent Classification

Binary classifier predicting whether a given incident is violent (homicide, rape, robbery, aggravated assault, weapons, kidnapping).

In [1]:
# bootstrap: make src importable
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from pathlib import Path
from src import config, classification
inc = pd.read_parquet(config.FEATURES_DIR / 'incident_features.parquet')
print(inc.shape)
inc['is_violent'].mean()

(955339, 27)


0.16732699073313242

## Train logistic regression + LightGBM

In [2]:
results = classification.train_classifier(inc, target='is_violent', models=('logreg','lgbm'), sample=200_000)
for name, r in results.items():
    print(f'{name:8s}  ACC={r.accuracy:.3f}  F1={r.f1_pos:.3f}  AUC={r.roc_auc:.3f}  PR-AUC={r.pr_auc:.3f}')

02:07:35 | INFO    | src.classification | [start] train logreg -> is_violent


C:\Users\cemil\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


02:07:37 | INFO    | src.classification | [done ] train logreg -> is_violent in 1.60s


02:07:37 | INFO    | src.classification | [logreg] acc=0.823  f1=0.650  AUC=0.915  PR-AUC=0.593


02:07:37 | INFO    | src.classification | [start] train lgbm -> is_violent


02:07:40 | INFO    | src.classification | [done ] train lgbm -> is_violent in 2.48s


02:07:40 | INFO    | src.classification | [lgbm] acc=0.826  f1=0.654  AUC=0.923  PR-AUC=0.620


logreg    ACC=0.823  F1=0.650  AUC=0.915  PR-AUC=0.593
lgbm      ACC=0.826  F1=0.654  AUC=0.923  PR-AUC=0.620


## Detailed classification report (LightGBM)

In [3]:
print(results['lgbm'].report)

              precision    recall  f1-score   support

           0       1.00      0.79      0.88     33326
           1       0.49      0.98      0.65      6674

    accuracy                           0.83     40000
   macro avg       0.74      0.89      0.77     40000
weighted avg       0.91      0.83      0.85     40000



## Top features driving the prediction

In [4]:
classification.feature_importances(results['lgbm'].pipeline, top=15)

,feature,importance
0,victim_age,3716
1,lon,3701
2,lat,3598
3,hour,1458
4,report_lag_days,1328
5,hour_cos,1195
6,month,1152
7,month_cos,960
8,hour_sin,918
9,dow,877


## Persisted metrics from the pipeline run

In [5]:
metrics = json.loads(Path('../reports/classification_metrics.json').read_text())
metrics

{'is_violent__logreg': {'accuracy': 0.822975,
  'f1_pos': 0.6504073068378178,
  'roc_auc': 0.9146141923473688,
  'pr_auc': 0.5929947689804309},
 'is_violent__lgbm': {'accuracy': 0.8263,
  'f1_pos': 0.6538806416259838,
  'roc_auc': 0.9229188520065964,
  'pr_auc': 0.6196398005390986},
 'is_arrest__logreg': {'accuracy': 0.676075,
  'f1_pos': 0.266266493006399,
  'roc_auc': 0.7123769621993127,
  'pr_auc': 0.18396469342273183},
 'is_arrest__lgbm': {'accuracy': 0.682575,
  'f1_pos': 0.27943930537426936,
  'roc_auc': 0.7407440085318164,
  'pr_auc': 0.20507539117667806}}